# NeuroVLM PubMed Semantic Evaluation Baseline

This notebook evaluates the pretrained NeuroVLM MLP baseline on the official PubMed split. Exact PubMed paper retrieval is kept as `exact_pmid_retrieval`, but semantic ranking is the main comparison target: network labels, PubMed MeSH terms, and semantic-neighborhood paper retrieval.

Run this first and keep the output directory as the MLP baseline reference when comparing ALE3DCNN, CoordGNN, CoordDeepSet, ALEFlatMLP, and DiFuMo GAT runs.

## 1. Imports / Device

In [1]:
from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "src").exists() else next(
    (p for p in CWD.parents if (p / "src").exists()),
    CWD,
)
repo_root = REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if (REPO_ROOT / "src").exists() and str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
print("repo_root:", REPO_ROOT)
print("device:", device)
print("torch:", torch.__version__)


repo_root: /Users/borng/code/lab_work/neurovlm
device: mps
torch: 2.10.0


## 2. Config

In [2]:
from neurovlm.semantic_evaluation import find_default_mesh_json

RESOURCE_DIR = REPO_ROOT / "experiments" / "evaluation_resources"
EVAL_RESOURCE_DIR = RESOURCE_DIR
OUT_DIR = REPO_ROOT / "experiments" / "runs" / "neurovlm_pubmed_semantic_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

KS = (1, 5, 10, 50)
BATCH_SIZE = 4096
TEXT_EMBED_BATCH_SIZE = 64
RANDOM_SPLIT_SEED = 0

RUN_NETWORK_LABELING = True
RUN_MESH_TERM_RANKING = True
RUN_SEMANTIC_NEIGHBOR_RETRIEVAL = True

NETWORK_LABELS_CSV = RESOURCE_DIR / "networks_labels" / "network_test_set_labels.csv"
DEFAULT_MESH_PMID_JSON = RESOURCE_DIR / "mesh_kg" / "mesh_annotations.json"
MESH_PMID_JSON = DEFAULT_MESH_PMID_JSON if DEFAULT_MESH_PMID_JSON.exists() else find_default_mesh_json()
MESH_CANDIDATES_FROM_ALL_PMIDS = True
SEMANTIC_NEIGHBORS = 10

print("resource_dir:", RESOURCE_DIR)
print("out_dir:", OUT_DIR)
print("network_labels_csv:", NETWORK_LABELS_CSV if NETWORK_LABELS_CSV.exists() else "auto-detect")
print("mesh_pmid_json:", MESH_PMID_JSON)


resource_dir: /Users/borng/code/lab_work/neurovlm/experiments/evaluation_resources
out_dir: /Users/borng/code/lab_work/neurovlm/experiments/runs/neurovlm_pubmed_semantic_baseline
network_labels_csv: /Users/borng/code/lab_work/neurovlm/experiments/evaluation_resources/networks_labels/network_test_set_labels.csv
mesh_pmid_json: /Users/borng/code/lab_work/neurovlm/experiments/evaluation_resources/mesh_kg/mesh_annotations.json


## 3. Load PubMed Latents, Metadata, And Model

In [3]:
from neurovlm.core import NeuroVLM
from neurovlm.data import load_dataset
from neurovlm.retrieval_resources import (
    _load_latent_neuro,
    _load_latent_text,
    _proj_head_text_infonce,
)

brain_latent, brain_pmids = _load_latent_neuro()
text_latent, text_pmids = _load_latent_text()
pub_df = load_dataset("pubmed_text").copy()

brain_pmids = np.asarray(brain_pmids).astype(str)
text_pmids = np.asarray(text_pmids).astype(str)
pub_df["pmid"] = pub_df["pmid"].astype(str)

nvlm = NeuroVLM(datasets=["pubmed"], device=str(device))
proj_head_text = _proj_head_text_infonce().to(device).eval()

print("brain_latent:", tuple(brain_latent.shape), "brain_pmids:", len(brain_pmids))
print("text_latent :", tuple(text_latent.shape), "text_pmids :", len(text_pmids))
print("publications:", pub_df.shape, "columns:", list(pub_df.columns))
print("NeuroVLM datasets:", nvlm.datasets)

latent_specter2_adhoc.pt:   0%|          | 0.00/95.0M [00:00<?, ?B/s]

brain_latent: (29868, 384) brain_pmids: 29868
text_latent : (30826, 768) text_pmids : 30826
publications: (30826, 8) columns: ['pmid', 'pmcid', 'doi', 'name', 'description', 'train', 'test', 'val']
NeuroVLM datasets: ('pubmed',)


## 4. Align Brain/Text Rows By PMID

In [4]:
brain_lookup = {p: i for i, p in enumerate(brain_pmids)}
brain_rows, text_rows, shared_pmids = [], [], []
for text_i, pmid in enumerate(text_pmids):
    brain_i = brain_lookup.get(pmid)
    if brain_i is None:
        continue
    brain_rows.append(brain_i)
    text_rows.append(text_i)
    shared_pmids.append(pmid)

brain_rows = np.asarray(brain_rows, dtype=np.int64)
text_rows = np.asarray(text_rows, dtype=np.int64)
shared_pmids = np.asarray(shared_pmids).astype(str)
assert np.array_equal(brain_pmids[brain_rows], text_pmids[text_rows])

print("aligned pairs:", len(shared_pmids))
print("first 5 alignment checks:")
for i in range(min(5, len(shared_pmids))):
    print(i, "pmid=", shared_pmids[i], "brain_pmid=", brain_pmids[brain_rows[i]], "text_pmid=", text_pmids[text_rows[i]])

aligned pairs: 29868
first 5 alignment checks:
0 pmid= 1589767 brain_pmid= 1589767 text_pmid= 1589767
1 pmid= 8530552 brain_pmid= 8530552 text_pmid= 8530552
2 pmid= 8624678 brain_pmid= 8624678 text_pmid= 8624678
3 pmid= 8670634 brain_pmid= 8670634 text_pmid= 8670634
4 pmid= 8994101 brain_pmid= 8994101 text_pmid= 8994101


## 5. Use Official PubMed Split Columns

In [5]:
from neurovlm.semantic_evaluation import save_official_pubmed_splits

splits = save_official_pubmed_splits(
    pub_df,
    shared_pmids,
    OUT_DIR,
    random_state=RANDOM_SPLIT_SEED,
)

test_pmid_set = set(splits["test"].astype(str).tolist())
val_pmid_set = set(splits["val"].astype(str).tolist())
train_pmid_set = set(splits["train"].astype(str).tolist())

test_pos = np.flatnonzero(np.asarray([pmid in test_pmid_set for pmid in shared_pmids]))
val_pos = np.flatnonzero(np.asarray([pmid in val_pmid_set for pmid in shared_pmids]))
train_pos = np.flatnonzero(np.asarray([pmid in train_pmid_set for pmid in shared_pmids]))

assert len(test_pos) == len(splits["test"]), "Some official test PMIDs did not align to both brain and text latents."
print({"train": len(train_pos), "val": len(val_pos), "test": len(test_pos)})

PubMed split counts: {'train': 23894, 'val': 2987, 'test': 2987}
{'train': 23894, 'val': 2987, 'test': 2987}


## 6. Shared PubMed / MeSH / Semantic Evaluation

This cell uses the same shared evaluation implementation used by the ALE3DCNN best/global notebook. It writes exact PubMed retrieval, normalized-k recall curves, MeSH ranking, semantic-neighbor retrieval, and the main comparison summary row.

In [6]:
from experiments.semantic_model_eval import run_embedding_semantic_evaluations
from neurovlm.metrics import retrieval_ranks

# Query = PubMed brain latent; candidates = exact PubMed papers from the same official test split.
test_brain_raw = brain_latent[brain_rows[test_pos]].float()
test_pmids = shared_pmids[test_pos]
candidate_text_rows = text_rows[test_pos]
candidate_pmids = text_pmids[candidate_text_rows]
assert np.array_equal(test_pmids, candidate_pmids)

# Match NeuroVLM.to_text(project=True): brain_shared is the projected image/brain embedding.
result = nvlm.to_text(test_brain_raw, datasets=["pubmed"], project=True)
scores_all = result.scores_by_dataset["pubmed"]
sim = scores_all[candidate_text_rows].T.contiguous()
brain_shared = result.query_embeddings.float()

with torch.no_grad():
    text_shared = F.normalize(
        proj_head_text(text_latent[candidate_text_rows].float().to(device)).detach().cpu(),
        dim=1,
        eps=1e-8,
    )
raw_text_test = text_latent[candidate_text_rows].float()

semantic_summary = run_embedding_semantic_evaluations(
    model_name="pretrained_neurovlm_mlp",
    brain_embeddings=brain_shared,
    text_embeddings=text_shared,
    raw_text_embeddings=raw_text_test,
    pmids=test_pmids,
    text_projector=proj_head_text,
    out_dir=OUT_DIR,
    device=device,
    resource_dir=EVAL_RESOURCE_DIR,
    mesh_json=MESH_PMID_JSON,
    run_mesh=RUN_MESH_TERM_RANKING,
    run_semantic_neighbors=RUN_SEMANTIC_NEIGHBOR_RETRIEVAL,
    semantic_neighbors=SEMANTIC_NEIGHBORS,
    extra_summary={
        "evaluation_impl": "experiments.semantic_model_eval.run_embedding_semantic_evaluations",
        "embedding_space": "pretrained_neurovlm_infonce",
    },
)

# Backward-compatible aliases for older notes/scripts that expected these filenames.
exact_summary = json.loads((OUT_DIR / "exact_pmid_retrieval_metrics.json").read_text())
recall_curves = pd.read_csv(OUT_DIR / "exact_pmid_retrieval_curves.csv")
with open(OUT_DIR / "pubmed_test_recall_metrics.json", "w") as f:
    json.dump(exact_summary, f, indent=2)
recall_curves.to_csv(OUT_DIR / "pubmed_test_recall_curves.csv", index=False)

exact_b2p = exact_summary["brain_to_paper"]
exact_p2b = exact_summary["paper_to_brain"]
ranks = retrieval_ranks(sim).cpu().numpy()
rank_df = pd.DataFrame({
    "pmid": test_pmids,
    "rank": ranks,
    "hit@10": ranks <= 10,
    "true_score": sim.diag().cpu().numpy(),
    "top1_pos": sim.argmax(dim=1).cpu().numpy(),
    "top1_score": sim.max(dim=1).values.cpu().numpy(),
})
rank_df["top1_pmid"] = test_pmids[rank_df["top1_pos"].to_numpy()]
rank_df.to_csv(OUT_DIR / "exact_pmid_retrieval_ranks.csv", index=False)
rank_df.to_csv(OUT_DIR / "pubmed_test_brain_to_paper_ranks.csv", index=False)

print("saved shared PubMed/MeSH/semantic artifacts to", OUT_DIR.resolve())
display(pd.DataFrame([semantic_summary]))
display(rank_df.sort_values("rank").head(10))


There are adapters available but none are activated for the forward pass.
Encoding label texts: 100%|██████████| 142/142 [00:39<00:00,  3.63it/s]


saved shared PubMed/MeSH/semantic artifacts to /Users/borng/code/lab_work/neurovlm/experiments/runs/neurovlm_pubmed_semantic_baseline


,model,n_test_pmids,exact_pmid_paper_recall_curve_auc,exact_pmid_normalized_k_recall_curve_auc,exact_pmid_mean_paper_recall_curve_auc,exact_pmid_mean_normalized_k_recall_curve_auc,exact_pmid_recall@1,exact_pmid_recall@5,exact_pmid_recall@10,exact_pmid_recall@50,...,semantic_mrr,semantic_paper_style_recall_curve_auc,semantic_normalized_k_recall_curve_auc,n_text_neighbors,n_queries,evaluation_impl,embedding_space,macro_retrieval_normalized_k_recall_curve_auc,macro_normalized_k_recall_curve_auc,macro_retrieval_normalized_k_auc_components
0,pretrained_neurovlm_mlp,2987,0.833709,0.833709,0.835157,0.835157,0.010378,0.051892,0.09307,0.241379,...,0.10661,0.946697,0.946697,10,2987,experiments.semantic_model_eval.run_embedding_...,pretrained_neurovlm_infonce,0.89436,0.89436,"[exact_pmid_normalized_k_recall_curve_auc, mes..."


,pmid,rank,hit@10,true_score,top1_pos,top1_score,top1_pmid
1381,25061565,1,True,0.358555,1381,0.358555,25061565
2482,33414733,1,True,0.411015,2482,0.411015,33414733
598,20204074,1,True,0.388027,598,0.388027,20204074
2176,30798012,1,True,0.353785,2176,0.353785,30798012
271,17126370,1,True,0.405632,271,0.405632,17126370
469,19240659,1,True,0.364428,469,0.364428,19240659
1014,23046904,1,True,0.410907,1014,0.410907,23046904
2445,33192481,1,True,0.350599,2445,0.350599,33192481
23,11230097,1,True,0.395947,23,0.395947,11230097
1006,22993433,1,True,0.421701,1006,0.421701,22993433


## 7. Shared Network Labeling / Network Term Evaluation

This uses the same `run_network_labeling_from_embeddings` helper that ALE3DCNN runs use. The only model-specific part is encoding network maps with the pretrained NeuroVLM MLP baseline.

In [7]:
if RUN_NETWORK_LABELING:
    from experiments.semantic_model_eval import run_network_labeling_from_embeddings
    from neurovlm.retrieval_resources import _load_autoencoder, _load_masker, _proj_head_image_infonce
    from neurovlm.semantic_evaluation import encode_network_maps, load_network_maps, preprocess_network_maps

    masker = _load_masker()
    autoencoder = _load_autoencoder().to(device).eval()
    proj_head_image = _proj_head_image_infonce().to(device).eval()

    network_records = preprocess_network_maps(load_network_maps(), masker)
    network_embeddings = encode_network_maps(
        "neurovlm_mlp",
        network_records,
        {"masker": masker, "autoencoder": autoencoder, "proj_head_image": proj_head_image},
        batch_size=128,
        device=device,
    )
    network_metrics = run_network_labeling_from_embeddings(
        model_name="pretrained_neurovlm_mlp",
        network_embeddings=network_embeddings,
        network_records=network_records,
        text_projector=proj_head_text,
        out_dir=OUT_DIR,
        device=device,
        resource_dir=EVAL_RESOURCE_DIR,
    )
    display(pd.DataFrame([network_metrics]).drop(columns=["classification_report", "one_vs_rest_auc"], errors="ignore"))
else:
    from experiments.semantic_model_eval import mark_network_labeling_skipped

    mark_network_labeling_skipped(OUT_DIR, "RUN_NETWORK_LABELING is False")
    network_metrics = {"skipped": True}


Encoding network maps: 100%|██████████| 185/185 [00:27<00:00,  6.70it/s]
There are adapters available but none are activated for the forward pass.
Encoding label texts: 100%|██████████| 2/2 [00:00<00:00,  3.17it/s]


,n_network_maps,n_evaluated,n_skipped_unknown,accuracy,top_2_accuracy,top_3_accuracy,macro_auc,term_recall@1,term_recall@5,term_recall@10,...,network_term_recall@10,network_term_recall@20,network_term_map,network_term_mrr,network_term_ndcg@10,network_term_median_best_true_term_rank,network_term_n_candidate_terms,macro_retrieval_normalized_k_recall_curve_auc,macro_normalized_k_recall_curve_auc,macro_retrieval_normalized_k_auc_components
0,185,119,66,0.789916,0.92437,0.941176,0.962841,0.731092,0.907563,0.97479,...,0.97479,1.0,0.584364,0.806513,0.646349,1.0,118,0.918349,0.918349,"[exact_pmid_normalized_k_recall_curve_auc, mes..."


## 8. Final Baseline Summary Row

The summary row is now produced by the same shared evaluator used by ALE3DCNN and then updated with network metrics.

In [8]:
summary_row = json.loads((OUT_DIR / "main_comparison_summary_row.json").read_text())
summary_df = pd.DataFrame([summary_row])
summary_df.to_csv(OUT_DIR / "main_comparison_summary_row.csv", index=False)
with open(OUT_DIR / "main_comparison_summary_row.json", "w") as f:
    json.dump(summary_row, f, indent=2)

important_cols = [
    "model",
    "n_test_pmids",
    "exact_pmid_paper_recall_curve_auc",
    "exact_pmid_normalized_k_recall_curve_auc",
    "exact_pmid_mean_normalized_k_recall_curve_auc",
    "exact_pmid_recall@10",
    "mesh_normalized_k_recall_curve_auc",
    "mesh_recall@10",
    "semantic_recall@10",
    "semantic_recall@50",
    "semantic_paper_style_recall_curve_auc",
    "semantic_normalized_k_recall_curve_auc",
    "macro_retrieval_normalized_k_recall_curve_auc",
    "macro_normalized_k_recall_curve_auc",
    "network_accuracy",
    "network_macro_auc",
    "network_term_normalized_k_recall_curve_auc",
    "network_term_recall@10",
    "network_term_mrr",
    "network_term_ndcg@10",
]
available_cols = [c for c in important_cols if c in summary_df.columns]
print("main summary:", OUT_DIR / "main_comparison_summary_row.csv")
display(summary_df[available_cols])


main summary: /Users/borng/code/lab_work/neurovlm/experiments/runs/neurovlm_pubmed_semantic_baseline/main_comparison_summary_row.csv


,model,n_test_pmids,exact_pmid_paper_recall_curve_auc,exact_pmid_normalized_k_recall_curve_auc,exact_pmid_mean_normalized_k_recall_curve_auc,exact_pmid_recall@10,mesh_normalized_k_recall_curve_auc,mesh_recall@10,semantic_recall@10,semantic_recall@50,semantic_paper_style_recall_curve_auc,semantic_normalized_k_recall_curve_auc,macro_retrieval_normalized_k_recall_curve_auc,macro_normalized_k_recall_curve_auc,network_accuracy,network_macro_auc,network_term_normalized_k_recall_curve_auc,network_term_recall@10,network_term_mrr,network_term_ndcg@10
0,pretrained_neurovlm_mlp,2987,0.833709,0.833709,0.835157,0.09307,0.902675,0.17551,0.231336,0.486441,0.946697,0.946697,0.918349,0.918349,0.789916,0.962841,0.990315,0.97479,0.806513,0.646349


## 12. Single PMID Sanity Check

In [9]:
SINGLE_PMID = None
TOP_K = 20

if SINGLE_PMID is None:
    query_pos = 0
else:
    matches = np.flatnonzero(test_pmids == str(SINGLE_PMID))
    if len(matches) == 0:
        raise ValueError(f"PMID {SINGLE_PMID} is not in the PubMed test split.")
    query_pos = int(matches[0])

query_pmid = str(test_pmids[query_pos])
query_brain = brain_latent[brain_rows[test_pos[query_pos]]].float().unsqueeze(0)
single_result = nvlm.to_text(query_brain, datasets=["pubmed"], project=True)
single_scores_all = single_result.scores_by_dataset["pubmed"][:, 0]
single_scores_test = single_scores_all[candidate_text_rows]
order = torch.argsort(single_scores_test, descending=True)
true_rank = int((order == query_pos).nonzero(as_tuple=True)[0].item()) + 1

true_row = pub_df.loc[pub_df["pmid"] == query_pmid].head(1).copy()
print("Query/test PMID:", query_pmid)
print("True paper rank among PubMed test candidates:", true_rank, "of", len(test_pmids))
display(true_row.T)

top_positions = order[:TOP_K].cpu().numpy()
top_pmids = test_pmids[top_positions]
top_scores = single_scores_test[top_positions].cpu().numpy()
top_df = pd.DataFrame({
    "rank": np.arange(1, len(top_positions) + 1),
    "pmid": top_pmids,
    "score": top_scores,
    "is_true_paper": top_pmids == query_pmid,
})
meta_cols = [c for c in ["pmid", "title", "name", "abstract", "description"] if c in pub_df.columns]
top_df = top_df.merge(pub_df[meta_cols], on="pmid", how="left")
display(top_df)

Query/test PMID: 8994101
True paper rank among PubMed test candidates: 22 of 2987


,26865
pmid,8994101
pmcid,NaN
doi,None
name,Acoustic neuroma: correlations between morphol...
description,Forty-two patients with acoustic neuroma (AN) ...
train,False
test,True
val,False


,rank,pmid,score,is_true_paper,name,description
0,1,21329758,0.520293,False,A data-driven framework for neural field model...,This paper presents a framework for creating n...
1,2,19349231,0.495528,False,Investigating the benefits of multi-echo EPI f...,Functional MRI studies on humans with BOLD con...
2,3,18761038,0.483327,False,An open-source hardware and software system fo...,Simultaneous recording of electrophysiology an...
3,4,19778619,0.463279,False,Estimating the transfer function from neuronal...,Previous studies using combined electrical and...
4,5,31470126,0.461352,False,SpinDoctor: A MATLAB toolbox for diffusion MRI...,The complex transverse water proton magnetizat...
5,6,34964194,0.450162,False,7T versus 3T MRI in the presurgical evaluation...,MRI has a crucial role in presurgical evaluati...
6,7,22328419,0.434059,False,Investigating causality between interacting br...,"In this work, we investigate the feasibility t..."
7,8,18515149,0.422854,False,Bayesian estimation of synaptic physiology fro...,We describe a Bayesian inference scheme for qu...
8,9,28559192,0.421606,False,Spatio-temporal TGV denoising for ASL perfusio...,In arterial spin labeling (ASL) a perfusion we...
9,10,27693612,0.419230,False,BrainNetCNN: Convolutional neural networks for...,"We propose BrainNetCNN, a convolutional neural..."
